In [1]:
from BinLattice import *
from Lattice import *
from LatticeUtils import *
from DiscForm import *
from IntVectors import *
from Vinberg import *
from FPSearch import *
from VSearch import *
from Allcock import *
from Circle import *
from Commons import *
from Genus import *
import fp_search_cpp
import vsearch_cpp

In [ ]:
from IPython.display import display, Math

L = D_lat(4)
G = Genus(L)
display(Math(G.str()))

<IPython.core.display.Math object>

In [ ]:
G = imat([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
          [0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1],
          [0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1],
          [0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1],
          [0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1]])
gens = [[2 * int(i == j) for i in range(16)] for j in range(16)]
for v in product(range(2), repeat=5):
    gens.append([x % 2 for x in (imat(v) @ G).tolist()])
basis = Lattice.image(imat(gens))
K = Lattice(16, basis @ basis.transpose() // 2)
L = K(-1)
print(L.info())
primes = factorint(L.exp)
print(primes)
for p in primes:
    print(p, ":", symbol(L.A, p, primes[p]))

Even lattice of signature (0, 16), discriminant 64 and exponent 2
Discriminant group: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2]
{2: 1}
2 : {1: (0, 0, 1, 6), 0: (0, 0, 1, 10)}


In [ ]:
with open('out', "r") as f:
    lattices = [re.findall(r'-?\d+', line.strip()) for line in f.readlines()]
    for j, l in enumerate(lattices[:100]):
        L = Lattice(3, [[int(x) for x in l[i:i+3]] for i in range(0, 9, 3)])
        print(L.info())
        primes = factorint(L.exp)
        for p in primes:
            print(p, ":", symbol(L.A, p, primes[p]))

In [ ]:
M = Leech_lat()
A, T = M.lll(transform=True)
FPS = fp_search_cpp.FPSearch(np.array(A, dtype=float), np.zeros(M.rank, dtype=float), 0, 4.5)
vecs = FPS.search_all()
print(len(vecs))

196560


In [2]:
def cusp_walls(L: Lattice, v: IMat, base: IMat) -> List[IMat]:
    if L.signature[0] != 1:
        raise ValueError("The lattice must be of signature (1, n)")
    if L.square(v) != 0:
        raise ValueError("The vector v must be isotropic")
    if L.square(base) <= 0:
        raise ValueError("The base vector must be positive")
    v = imat(v) // math.gcd(*v.tolist())
    if L.product(v, base) < 0:
        v = -v
    basis = L.subquotient(L.complement(v))
    dual, a = L.dual_vec(v)
    if nrows(basis) != L.rank - 2:
        raise ValueError("Error constructing subquotient")
    M = Lattice(nrows(basis), L.batch_prod(basis, basis))
    A, T = M.lll(transform=True)
    basis = imat(list_rows(T @ basis))
    M = Lattice(M.rank, A)
    FPS = fp_search_cpp.FPSearch(np.array(-A, dtype=float), np.zeros(M.rank, dtype=float), 0, 2 * M.exp + 1)
    vecs = FPS.search_all()
    roots = []
    for u in vecs:
        if not M.is_root(u):
            continue
        w = imat(u) @ basis
        b = L.square(w)
        c, x, _, _, _ = euclid(2 * a, b)
        y = 2 * L.product(dual, w)
        if y % c != 0:
            continue
        w = w - (y // c) * x * v
        delta = abs(b // c)
        x, y = divmod(-L.product(w, base), (delta * L.product(v, base)))
        if y == 0:
            shift = [x - 1, x, x + 1]
        else:
            shift = [x, x + 1]
        for k in shift:
            roots.append(w + k * delta * v)
    return roots

In [3]:
def geodesic_walls(L: Lattice, basis: IMat, base: IMat, nshifts: int = 5) -> List[IMat]:
    if nrows(basis) != 2:
        raise ValueError("The sublattice has to be of rank 2")
    M = L.batch_prod(basis, basis)
    B = BinLattice(M[0, 0], M[1, 1], M[0, 1])
    if B.signature != (1, 1):
        # raise ValueError("The sublattice has to be of signature (1, 1)")
        return []
    shifts = [basis]
    if B.shift is not None:
        S = imat_diag([1] * 2)
        for i in range(nshifts):
            S = S @ B.shift
            shifts.append(S @ basis)            
        S = imat_diag([1] * 2)
        T, _ = inv2x2(B.shift)
        for i in range(nshifts):
            S = S @ T
            shifts.append(S @ basis)
    vecs = B.list_negative(2 * L.exp)
    roots = []
    for s in vecs.keys():
        if (2 * L.exp) % s != 0:
            continue
        for v, T in product(vecs[s], shifts):
            w = imat(v) @ T
            if L.square(w) >= 0:
                raise ValueError("Positive root!")
            if not L.is_root(w):
                continue
            d = L.product(w, base)
            if d == 0:
                continue
            if d < 0:
                w = -w
            roots.append((w, Fraction(d * d, abs(int(s)))))
    roots.sort(key=lambda x: x[1])
    if len(roots) == 0:
        return []
    walls = [roots[0][0]]
    for r in roots:
        if L.product(r[0], walls[0]) > 0:
            walls.append(r[0])
            break
    return walls

In [17]:
# rank = 9
# L = Lattice(rank, [[1 - 2 * int(i == j) for j in range(rank)] for i in range(rank)])
# print(L.info())
# V = Vinberg(L, base=[1] * rank, h_batch=20, fps_batch=10000)
# walls = V.run(iterations=5)

# rank = 18
# L = I_lat(1, rank - 1)
# print(L.info())
# V = Vinberg(L, base=[1] + [0] * (rank - 1), h_batch=1, fps_batch=10000)
# walls = V.run(iterations=3)

L = D_lat(20)(-1) + U_lat()
print(L.info())
V = Vinberg(L, base=[0] * 20 + [1, 1], h_batch=1, fps_batch=10000)
walls = V.run(iterations=4)

Even lattice of signature (1, 21), discriminant -4 and exponent 2
Discriminant group: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2]


In [ ]:
squares = [(ray * L.A_fl * ray.transpose())[0, 0] for ray in V.rays]
for a, (r, s) in enumerate(zip(V.rays, squares)):
    if s >= 0:
        continue
    ray, _ = flq2imat(r)
    new_walls = []
    if L.is_root(ray):
        new_walls.append(ray if V.L.product(ray, V.base) > 0 else -ray)
    adj = [w[0] for w in V.active_walls if L.product(ray, w[0]) == 0]
    for i in range(len(adj)):
        compl = L.complement([adj[j] for j in range(len(adj)) if i != j])
        if nrows(compl) != 2:
            continue
        M = L.batch_prod(compl, compl)
        Lcompl = BinLattice(M[0, 0], M[1, 1], M[0, 1])
        if Lcompl.zero:
            for v in Lcompl.zero:
                w = imat(v) @ compl
                if L.product(w, V.base) < 0:
                    continue
                new_walls.extend(cusp_walls(L, w, V.base))
        new_walls.extend(geodesic_walls(L, compl, V.base, nshifts=3))
    V.update_walls(new_walls)
    print(f"{a}-th ray of {len(V.rays)}; {len(V.active_walls)} active walls", end='\r')
print()
print("Updating rays...")
print(V.update_rays())

In [ ]:
print("Hello")

In [4]:
L = D_lat(20)(-1) + U_lat()
print(L.info())
V = Vinberg(L, base=[0] * 20 + [1, 1], h_batch=1, fps_batch=10000)
walls = V.run(root_batch=10, max_iterations=3)
print(len(walls), 'walls found')
for w in walls:
    print(w, L.square(w))
rays, lines = get_extremal_rays(walls, L.A)
print(len(rays), len(lines))
squares = [(fl.fmpq_mat(1, L.rank, ray) * L.A * fl.fmpq_mat(L.rank, 1, ray))[0, 0] for ray in rays]
for r, s in zip(rays, squares):
    print(r, s)

Even lattice of signature (1, 21), discriminant -4 and exponent 2
Discriminant group: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2]
Iteration 3; 22 walls
Reached the maximum number of iterations
22 walls found
[0, 0, 0, 0, 0, 0, -1, -1, -2, -2, -2, -2, -2, -2, -2, -2, -2, -2, -1, -1, 0, 0] -2
[0, 0, 0, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 0, 0] -2
[0, 0, 0, 0, 0, -1, -1, -1, -1, -1, -1, -1, -2, -2, -2, -2, -2, -2, -1, -1, 0, 0] -2
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0] -2
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0] -2
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, -1, 0, 0, 0, 0, 0] -2
[0, 0, 0, 0, 0, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 0, 0] -2
[0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0] -2
[0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 1, 1, 0, 0] -2
[0, 0, 0, 0, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -2, -1, -1, 0, 0] -2
[0, 0, 0, 0, 0, 0, 0, 0

In [ ]:
for r, s in zip(rays, squares):
    if s >= 0:
        continue
    ray, _ = fl.fmpq_mat(1, L.rank, r).numer_denom()
    print(ray, L.is_root(ray))
    adj = [w for w in walls if L.product(ray, w) == 0]
    for i in range(len(adj)):
        compl = L.complement([walls[j] for j in range(len(adj)) if i != j])
        M = L.batch_prod(compl, compl)
        Lcompl = BinLattice(M[0][0], M[1][1], M[0][1])
        print(Lcompl.info())

In [ ]:
L = I_lat(1, 21)
basis = L.even_sublattice()
L = Lattice(22, L.batch_prod(basis, basis))
print(L.info())
D = DiscForm(L)
iso = D.list_max_isospaces()
print(D.Ared, D.iso)

compl = [[[2, 0], [0, 6]], [[4, 2], [2, 4]]]
for b in compl:
    C = Lattice(2, b)
    print(C.info())
    D = DiscForm(L + C)
    print(D.Ared)
    iso = D.list_max_isospaces()
    for s in iso:
        M = D.overlattice(s)
        if M.parity == 0:
            print(M.info())
            DD = DiscForm(M)
            print(DD.iso)
            print(DD.Ared)